In [ ]:
import os
import squidpy as sq
import scanpy as sc
import anndata as ad
import pandas as pd
from adpbulk import ADPBulk
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix

import warnings
warnings.filterwarnings("ignore")

In [ ]:
in_dir = "../../data"
out_dir = "../../results/11_compare_with_GSE294965/01_viz_data"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
adata = sc.read_h5ad(f"{in_dir}/GSE294965_processed_data.h5ad")

In [ ]:
adata

In [ ]:
adata.obs.columns

In [ ]:
adata.X = adata.layers["counts"].copy()

In [ ]:
# initialize the object
adpb = ADPBulk(adat=adata, groupby=["sample"], layer="counts")

counts = csr_matrix(adpb.fit_transform())
sample_meta = adpb.get_meta().set_index("sample")

counts.index = sample_meta.index

adata_bulk = ad.AnnData(X=counts, 
                        obs=sample_meta, 
                        var=adata.var.copy())
adata_bulk.layers["counts"] = adata_bulk.X.copy()

adata_bulk.var_names_make_unique()


In [ ]:
adata

In [ ]:
## add group info
group_dict = adata.obs.set_index("sample")["label"].to_dict()

adata_bulk.obs['group'] = adata_bulk.obs_names.map(group_dict)

In [ ]:
adata_bulk.obs

In [ ]:
adata_bulk.layers["counts"] = adata_bulk.X.copy()

In [ ]:
sc.pp.normalize_total(adata_bulk, target_sum=1e6)
sc.pp.log1p(adata_bulk)

In [ ]:
# for gene in adata_bulk.var_names:
#     fig, ax = plt.subplots(figsize=(6, 4))
#     sc.pl.violin(adata_bulk, keys=gene, 
#                  groupby="group", 
#                  use_raw=False,
#                  rotation=45, 
#                  ax=ax, 
#                  show=False)
#     plt.title(f"Expression of {gene} across groups")
#     plt.tight_layout()
#     plt.savefig(f"{out_dir}/violin_{gene}.png")
#     plt.close()

In [ ]:
adata_bulk.write_h5ad(f"{out_dir}/pseudo_bulk_adata.h5ad")